In [5]:
import psycopg
import numpy as np

try:
    with psycopg.connect(
        "dbname=chatbot_rag user=postgres password=t17@ACHA host=localhost port=5432"
    ) as conn:
        with conn.cursor() as cur:
            # Créer la table
            cur.execute("""
                CREATE TABLE IF NOT EXISTS embeddings(
                    id SERIAL PRIMARY KEY,
                    corpus TEXT NOT NULL,
                    embedding FLOAT8[]
                );
            """)
            
            # Insérer des données
            corpus = "test corpus"
            embedding = np.random.rand(5).tolist()
            
            cur.execute("""
                INSERT INTO embeddings (corpus, embedding) 
                VALUES (%s, %s)
            """, (corpus, embedding))
            
            # Le commit est automatique avec le context manager
            print("✓ Table créée et données insérées avec succès")
            
except psycopg.OperationalError as e:

    print(f"❌ Erreur de connexion : {e}")
    
    print("Vérifiez que PostgreSQL est déma-rré sur le port 5432")
except Exception as e:
    print(f"❌ Erreur : {e}")

✓ Table créée et données insérées avec succès


In [6]:
from sentence_transformers import SentenceTransformer
import psycopg
import numpy as np
import json
from typing import List, Dict

print("✅ Tous les modules importés avec succès!")

c:\Users\HP\Desktop\Docs M2\Machine learning Dr Bisyande\Chatbot_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tous les modules importés avec succès!


In [8]:
# ===== CONFIGURATION =====

# Configuration base de données
DB_CONFIG = {
    "dbname": "chatbot_rag",
    "user": "postgres",
    "password": "t17@ACHA",
    "host": "localhost",
    "port": "5432",
}

# Modèle d'embeddings
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# Paramètres
TOP_K_RESULTS = 3  # Nombre de résultats à retourner

print("✅ Configuration chargée")
print(f"📊 Modèle : {EMBEDDING_MODEL_NAME}")
print(f"🗄️  Base de données : {DB_CONFIG['dbname']}")

✅ Configuration chargée
📊 Modèle : paraphrase-multilingual-MiniLM-L12-v2
🗄️  Base de données : chatbot_rag


In [10]:
# ===== CHARGEMENT DU MODÈLE =====

print("📥 Chargement du modèle d'embeddings...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("✅ Modèle chargé avec succès!")

# Test rapide
test_text = "Bonjour, ceci est un test"
test_embedding = embedding_model.encode(test_text)
print(f"📐 Dimension des embeddings : {len(test_embedding)}")
print(f"🔢 Exemple : {test_embedding[:5]}")

📥 Chargement du modèle d'embeddings...
✅ Modèle chargé avec succès!
📐 Dimension des embeddings : 384
🔢 Exemple : [ 0.08716737  0.33370736 -0.0766759   0.04356514 -0.09947915]


In [12]:
# ===== SETUP BASE DE DONNÉES =====

def setup_database():
    """Créer une nouvelle table pour le projet RAG"""
    conn_string = f"dbname={DB_CONFIG['dbname']} user={DB_CONFIG['user']} password={DB_CONFIG['password']} host={DB_CONFIG['host']} port={DB_CONFIG['port']}"
    
    try:
        with psycopg.connect(conn_string) as conn:
            with conn.cursor() as cur:
                # Créer la nouvelle table
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS projet_themes(
                        id SERIAL PRIMARY KEY,
                        titre TEXT NOT NULL,
                        description TEXT NOT NULL,
                        domaine TEXT,
                        embedding FLOAT8[],
                        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                    );
                """)
                
                conn.commit()
                print("✅ Table 'projet_themes' créée avec succès")
                
                # Vérifier le nombre de lignes
                cur.execute("SELECT COUNT(*) FROM projet_themes")
                count = cur.fetchone()[0]
                print(f"📊 Nombre de thèmes existants : {count}")
                
                # Afficher les deux tables
                cur.execute("""
                    SELECT table_name 
                    FROM information_schema.tables 
                    WHERE table_schema = 'public'
                    ORDER BY table_name
                """)
                tables = cur.fetchall()
                print(f"\n📋 Tables dans la base de données :")
                for table in tables:
                    print(f"  - {table[0]}")
                
    except Exception as e:
        print(f"❌ Erreur : {e}")

# Exécuter le setup
setup_database()

✅ Table 'projet_themes' créée avec succès
📊 Nombre de thèmes existants : 6

📋 Tables dans la base de données :
  - embeddings
  - projet_themes


In [2]:
# ===== DONNÉES DE TEST =====

themes_projets = [
    {
        'titre': 'Système de recommandation de films avec filtrage collaboratif',
        'description': 'Développer un système qui recommande des films basés sur les préférences des utilisateurs similaires en utilisant des techniques de filtrage collaboratif et de factorisation matricielle.',
        'domaine': 'Machine Learning, Systèmes de recommandation, Filtrage collaboratif'
    },
    {
        'titre': 'Chatbot RAG pour support client automatisé',
        'description': 'Créer un assistant conversationnel intelligent qui utilise la recherche documentaire (RAG) pour répondre aux questions des clients en se basant sur une base de connaissances.',
        'domaine': 'NLP, RAG, Deep Learning, Traitement du langage naturel'
    },
    {
        'titre': 'Détection de fraude bancaire en temps réel',
        'description': 'Modèle de machine learning pour identifier les transactions frauduleuses en analysant les patterns de transactions et en détectant les anomalies.',
        'domaine': 'Machine Learning, Détection d\'anomalies, Finance, Sécurité'
    },
    {
        'titre': 'Classification d\'images médicales avec CNN',
        'description': 'Utiliser des réseaux de neurones convolutifs pour classifier des images médicales (rayons X, IRM) et aider au diagnostic précoce de maladies.',
        'domaine': 'Deep Learning, Computer Vision, Santé, CNN'
    },
    {
        'titre': 'Prédiction de séries temporelles pour la demande énergétique',
        'description': 'Modèle prédictif utilisant LSTM ou Transformer pour prévoir la consommation électrique en fonction de données historiques et météorologiques.',
        'domaine': 'Deep Learning, Time Series, LSTM, Énergie'
    },
    {
        'titre': 'Analyse de sentiment sur les réseaux sociaux',
        'description': 'Analyser les opinions et émotions exprimées dans les tweets ou posts en utilisant des techniques de NLP et de classification de texte.',
        'domaine': 'NLP, Analyse de sentiment, Classification, Réseaux sociaux'
    }
]

print(f"✅ {len(themes_projets)} thèmes de projets préparés")
for i, theme in enumerate(themes_projets, 1):
    print(f"{i}. {theme['titre']}")

✅ 6 thèmes de projets préparés
1. Système de recommandation de films avec filtrage collaboratif
2. Chatbot RAG pour support client automatisé
3. Détection de fraude bancaire en temps réel
4. Classification d'images médicales avec CNN
5. Prédiction de séries temporelles pour la demande énergétique
6. Analyse de sentiment sur les réseaux sociaux


In [13]:
# ===== FONCTION D'INDEXATION =====

def indexer_themes(themes: List[Dict]):
    """Indexer les thèmes dans la base de données avec leurs embeddings"""
    conn_string = f"dbname={DB_CONFIG['dbname']} user={DB_CONFIG['user']} password={DB_CONFIG['password']} host={DB_CONFIG['host']} port={DB_CONFIG['port']}"
    
    try:
        with psycopg.connect(conn_string) as conn:
            with conn.cursor() as cur:
                # Vider la table d'abord (pour éviter les doublons lors des tests)
                cur.execute("TRUNCATE TABLE projet_themes RESTART IDENTITY")
                print("🗑️  Table 'projet_themes' vidée")
                
                compteur = 0
                for theme in themes:
                    # Créer le texte à embedder (combinaison de tous les champs)
                    text = f"{theme['titre']} {theme['description']} {theme['domaine']}"
                    
                    # Générer l'embedding
                    print(f"⏳ Génération embedding pour : {theme['titre'][:50]}...")
                    embedding = embedding_model.encode(text).tolist()
                    
                    # Insérer dans la DB
                    cur.execute("""
                        INSERT INTO projet_themes (titre, description, domaine, embedding)
                        VALUES (%s, %s, %s, %s)
                    """, (theme['titre'], theme['description'], theme['domaine'], embedding))
                    
                    compteur += 1
                
                conn.commit()
                print(f"\n✅ {compteur} thèmes indexés avec succès dans 'projet_themes'!")
                
    except Exception as e:
        print(f"❌ Erreur lors de l'indexation : {e}")

# Indexer les thèmes
indexer_themes(themes_projets)

🗑️  Table 'projet_themes' vidée
⏳ Génération embedding pour : Système de recommandation de films avec filtrage c...
⏳ Génération embedding pour : Chatbot RAG pour support client automatisé...
⏳ Génération embedding pour : Détection de fraude bancaire en temps réel...
⏳ Génération embedding pour : Classification d'images médicales avec CNN...
⏳ Génération embedding pour : Prédiction de séries temporelles pour la demande é...
⏳ Génération embedding pour : Analyse de sentiment sur les réseaux sociaux...

✅ 6 thèmes indexés avec succès dans 'projet_themes'!


In [14]:
# ===== RECHERCHE SÉMANTIQUE =====

def rechercher_themes_pertinents(requete: str, top_k: int = 3):
    """Rechercher les thèmes les plus pertinents pour une requête"""
    
    print(f"\n🔍 Recherche pour : '{requete}'")
    print("⏳ Génération de l'embedding de la requête...")
    
    # Générer l'embedding de la requête
    query_embedding = embedding_model.encode(requete).tolist()
    
    conn_string = f"dbname={DB_CONFIG['dbname']} user={DB_CONFIG['user']} password={DB_CONFIG['password']} host={DB_CONFIG['host']} port={DB_CONFIG['port']}"
    
    try:
        with psycopg.connect(conn_string) as conn:
            with conn.cursor() as cur:
                # Récupérer tous les thèmes
                cur.execute("SELECT id, titre, description, domaine, embedding FROM projet_themes")
                results = cur.fetchall()
                
                print(f"📊 Comparaison avec {len(results)} thèmes...")
                
                # Calculer les similarités cosinus
                similarities = []
                for row in results:
                    theme_id, titre, description, domaine, db_embedding = row
                    
                    # Similarité cosinus
                    similarity = np.dot(query_embedding, db_embedding) / (
                        np.linalg.norm(query_embedding) * np.linalg.norm(db_embedding)
                    )
                    
                    similarities.append({
                        'id': theme_id,
                        'titre': titre,
                        'description': description,
                        'domaine': domaine,
                        'score': float(similarity)
                    })
                
                # Trier par pertinence décroissante
                similarities.sort(key=lambda x: x['score'], reverse=True)
                
                # Afficher les résultats
                print(f"\n✅ Top {top_k} thèmes les plus pertinents :\n")
                for i, theme in enumerate(similarities[:top_k], 1):
                    print(f"{i}. [{theme['score']:.4f}] {theme['titre']}")
                    print(f"   Domaine : {theme['domaine']}")
                    print()
                
                return similarities[:top_k]
                
    except Exception as e:
        print(f"❌ Erreur lors de la recherche : {e}")
        return []

# Test de la fonction
themes_trouves = rechercher_themes_pertinents(
    "Je veux faire un projet sur l'intelligence artificielle conversationnelle", 
    top_k=3
)


🔍 Recherche pour : 'Je veux faire un projet sur l'intelligence artificielle conversationnelle'
⏳ Génération de l'embedding de la requête...
📊 Comparaison avec 6 thèmes...

✅ Top 3 thèmes les plus pertinents :

1. [0.6416] Chatbot RAG pour support client automatisé
   Domaine : NLP, RAG, Deep Learning, Traitement du langage naturel

2. [0.3879] Classification d'images médicales avec CNN
   Domaine : Deep Learning, Computer Vision, Santé, CNN

3. [0.3425] Système de recommandation de films avec filtrage collaboratif
   Domaine : Machine Learning, Systèmes de recommandation, Filtrage collaboratif



In [15]:
# ===== TEST OLLAMA (CORRIGÉ) =====

import ollama

def tester_ollama():
    """Vérifier si Ollama est installé et fonctionnel"""
    try:
        # Essayer de lister les modèles disponibles
        models = ollama.list()
        print("✅ Ollama est installé et fonctionnel!")
        print(f"\n📋 Modèles disponibles :")
        
        if models and 'models' in models and len(models['models']) > 0:
            for model in models['models']:
                # Vérifier si 'name' existe, sinon utiliser 'model'
                model_name = model.get('name') or model.get('model', 'Unknown')
                print(f"  - {model_name}")
            return True
        else:
            print("  Aucun modèle installé")
            print("\n💡 Pour installer un modèle, ouvrez un terminal et tapez :")
            print("     ollama pull llama3.2:1b")
            return False
        
    except Exception as e:
        print("❌ Ollama n'est pas installé ou ne fonctionne pas")
        print(f"Erreur : {e}")
        print("\n📥 Installation :")
        print("1. Téléchargez Ollama sur https://ollama.com")
        print("2. Installez-le")
        print("3. Ouvrez un terminal et tapez : ollama pull llama3.2:1b")
        return False

# Tester Ollama
ollama_ok = tester_ollama()

✅ Ollama est installé et fonctionnel!

📋 Modèles disponibles :
  - llama3.2:1b


In [16]:
# ===== GÉNÉRATION DE RÉPONSES =====
from typing import List, Dict

def generer_reponse_avec_ollama(requete: str, themes_pertinents: List[Dict]):
    """Générer une réponse avec Ollama en utilisant les thèmes pertinents"""
    
    # Construire le contexte à partir des thèmes pertinents
    contexte = "\n\n".join([
        f"Thème {i+1}: {t['titre']}\n"
        f"Description: {t['description']}\n"
        f"Domaine: {t['domaine']}\n"
        f"Score de pertinence: {t['score']:.2f}"
        for i, t in enumerate(themes_pertinents)
    ])
    
    # Créer le prompt pour le LLM
    prompt = f"""Tu es un assistant qui aide les étudiants à choisir des thèmes de projets en Machine Learning.

Contexte - Thèmes de projets pertinents trouvés :
{contexte}

Question de l'étudiant : {requete}

Réponds de manière claire et structurée en :
1. Analysant les thèmes les plus pertinents
2. Expliquant pourquoi ils correspondent à la demande
3. Donnant des conseils pour choisir le meilleur projet

Réponds en français."""
    
    try:
        print("\n🤖 Génération de la réponse avec Ollama...")
        print("⏳ (Cela peut prendre quelques secondes...)\n")
        
        # Appel à Ollama
        response = ollama.chat(
            model='llama3.2:1b',
            messages=[{
                'role': 'user',
                'content': prompt
            }]
        )
        
        reponse = response['message']['content']
        print("✅ Réponse générée !\n")
        print("="*60)
        print(reponse)
        print("="*60)
        
        return reponse
        
    except Exception as e:
        print(f"❌ Erreur avec Ollama : {e}")
        print("\n💡 Réponse de fallback (sans LLM) :")
        
        # Réponse simplifiée sans LLM
        reponse_fallback = f"Basé sur votre recherche, voici les thèmes les plus pertinents :\n\n"
        for i, theme in enumerate(themes_pertinents, 1):
            reponse_fallback += f"{i}. {theme['titre']} (Score: {theme['score']:.2f})\n"
            reponse_fallback += f"   {theme['description']}\n\n"
        
        print(reponse_fallback)
        return reponse_fallback


# ===== TEST DE LA FONCTION =====

# D'abord, chercher les thèmes pertinents
print("="*70)
print("📋 ÉTAPE 1 : Recherche des thèmes pertinents")
print("="*70)

themes_trouves = rechercher_themes_pertinents(
    "Je veux faire un projet sur l'intelligence artificielle conversationnelle", 
    top_k=3
)

# Ensuite, générer la réponse
print("\n" + "="*70)
print("📋 ÉTAPE 2 : Génération de la réponse avec le LLM")
print("="*70)

if ollama_ok and themes_trouves:
    generer_reponse_avec_ollama(
        "Je veux faire un projet sur l'intelligence artificielle conversationnelle",
        themes_trouves
    )
elif not ollama_ok:
    print("\n⚠️ Ollama n'est pas disponible, on utilise la réponse simplifiée")
    if themes_trouves:
        generer_reponse_avec_ollama(
            "Je veux faire un projet sur l'intelligence artificielle conversationnelle",
            themes_trouves
        )
else:
    print("❌ Aucun thème trouvé")

📋 ÉTAPE 1 : Recherche des thèmes pertinents

🔍 Recherche pour : 'Je veux faire un projet sur l'intelligence artificielle conversationnelle'
⏳ Génération de l'embedding de la requête...
📊 Comparaison avec 6 thèmes...

✅ Top 3 thèmes les plus pertinents :

1. [0.6416] Chatbot RAG pour support client automatisé
   Domaine : NLP, RAG, Deep Learning, Traitement du langage naturel

2. [0.3879] Classification d'images médicales avec CNN
   Domaine : Deep Learning, Computer Vision, Santé, CNN

3. [0.3425] Système de recommandation de films avec filtrage collaboratif
   Domaine : Machine Learning, Systèmes de recommandation, Filtrage collaboratif


📋 ÉTAPE 2 : Génération de la réponse avec le LLM

🤖 Génération de la réponse avec Ollama...
⏳ (Cela peut prendre quelques secondes...)

✅ Réponse générée !

Bonjour étudiant,

Nous avons identifié trois thèmes de projets qui correspondent parfaitement à votre demande d'intelligence artificielle conversationnelle :

**Thème 1 : Chatbot RAG pour suppor

In [17]:
# ===== CHATBOT RAG COMPLET =====

def chatbot_rag(requete: str, top_k: int = 3, use_llm: bool = True):
    """Fonction principale du chatbot RAG"""
    
    print("\n" + "="*70)
    print("🤖 CHATBOT RAG - Validation de Thèmes de Projets")
    print("="*70)
    
    # 1. Recherche sémantique
    themes_pertinents = rechercher_themes_pertinents(requete, top_k)
    
    if not themes_pertinents:
        print("❌ Aucun thème trouvé")
        return
    
    # 2. Génération de réponse
    if use_llm and ollama_ok:
        reponse = generer_reponse_avec_ollama(requete, themes_pertinents)
    else:
        # Réponse simple sans LLM
        reponse = f"\n📋 Thèmes pertinents pour votre recherche :\n\n"
        for i, theme in enumerate(themes_pertinents, 1):
            reponse += f"{i}. {theme['titre']} (Pertinence: {theme['score']:.2f})\n"
            reponse += f"   📖 {theme['description']}\n"
            reponse += f"   🏷️  {theme['domaine']}\n\n"
        print(reponse)
    
    return themes_pertinents

# ===== TESTS =====

print("\n🧪 TEST 1 : Projet sur l'IA conversationnelle")
chatbot_rag("Je veux faire un projet sur l'intelligence artificielle conversationnelle")

print("\n\n🧪 TEST 2 : Projet sur la finance")
chatbot_rag("Quel projet en machine learning pour la finance?")

print("\n\n🧪 TEST 3 : Projet sur la santé")
chatbot_rag("Je m'intéresse à l'application du deep learning dans le domaine médical")


🧪 TEST 1 : Projet sur l'IA conversationnelle

🤖 CHATBOT RAG - Validation de Thèmes de Projets

🔍 Recherche pour : 'Je veux faire un projet sur l'intelligence artificielle conversationnelle'
⏳ Génération de l'embedding de la requête...
📊 Comparaison avec 6 thèmes...

✅ Top 3 thèmes les plus pertinents :

1. [0.6416] Chatbot RAG pour support client automatisé
   Domaine : NLP, RAG, Deep Learning, Traitement du langage naturel

2. [0.3879] Classification d'images médicales avec CNN
   Domaine : Deep Learning, Computer Vision, Santé, CNN

3. [0.3425] Système de recommandation de films avec filtrage collaboratif
   Domaine : Machine Learning, Systèmes de recommandation, Filtrage collaboratif


🤖 Génération de la réponse avec Ollama...
⏳ (Cela peut prendre quelques secondes...)

✅ Réponse générée !

Bonjour ! Je suis ravi de vous aider.

### Analyse des Thèmes pertinents

1. **Thème 1 : Chatbot RAG pour support client automatisé**
 * Description : Créer un assistant conversationnel intellig

[{'id': 4,
  'titre': "Classification d'images médicales avec CNN",
  'description': 'Utiliser des réseaux de neurones convolutifs pour classifier des images médicales (rayons X, IRM) et aider au diagnostic précoce de maladies.',
  'domaine': 'Deep Learning, Computer Vision, Santé, CNN',
  'score': 0.5258958839004488},
 {'id': 2,
  'titre': 'Chatbot RAG pour support client automatisé',
  'description': 'Créer un assistant conversationnel intelligent qui utilise la recherche documentaire (RAG) pour répondre aux questions des clients en se basant sur une base de connaissances.',
  'domaine': 'NLP, RAG, Deep Learning, Traitement du langage naturel',
  'score': 0.3665068558504082},
 {'id': 3,
  'titre': 'Détection de fraude bancaire en temps réel',
  'description': 'Modèle de machine learning pour identifier les transactions frauduleuses en analysant les patterns de transactions et en détectant les anomalies.',
  'domaine': "Machine Learning, Détection d'anomalies, Finance, Sécurité",
  's

In [19]:
# ===== CELLULE 11 : Évaluation du système =====

def evaluer_systeme():
    """Évaluer la performance du système RAG"""
    
    # Questions de test avec leurs réponses attendues
    tests = [
        {
            'question': "Je veux faire un chatbot intelligent",
            'theme_attendu': "Chatbot RAG"
        },
        {
            'question': "Comment détecter les fraudes dans les banques?",
            'theme_attendu': "Détection de fraude"
        },
        {
            'question': "Projet sur la classification d'images médicales",
            'theme_attendu': "Classification d'images médicales"
        }
    ]
    
    resultats = []
    for test in tests:
        themes = rechercher_themes_pertinents(test['question'], top_k=1)
        if themes:
            premier_theme = themes[0]
            succes = test['theme_attendu'].lower() in premier_theme['titre'].lower()
            resultats.append({
                'question': test['question'],
                'attendu': test['theme_attendu'],
                'obtenu': premier_theme['titre'],
                'score': premier_theme['score'],
                'succes': succes
            })
    
    # Afficher les résultats
    print("\n📊 ÉVALUATION DU SYSTÈME")
    print("="*70)
    taux_succes = sum(1 for r in resultats if r['succes']) / len(resultats) * 100
    print(f"Taux de réussite : {taux_succes:.1f}%\n")
    
    for i, r in enumerate(resultats, 1):
        symbole = "✅" if r['succes'] else "❌"
        print(f"{i}. {symbole} Score: {r['score']:.4f}")
        print(f"   Question : {r['question']}")
        print(f"   Attendu  : {r['attendu']}")
        print(f"   Obtenu   : {r['obtenu']}\n")
    
    return resultats

# Exécuter l'évaluation
evaluer_systeme()


🔍 Recherche pour : 'Je veux faire un chatbot intelligent'
⏳ Génération de l'embedding de la requête...
📊 Comparaison avec 6 thèmes...

✅ Top 1 thèmes les plus pertinents :

1. [0.6272] Chatbot RAG pour support client automatisé
   Domaine : NLP, RAG, Deep Learning, Traitement du langage naturel


🔍 Recherche pour : 'Comment détecter les fraudes dans les banques?'
⏳ Génération de l'embedding de la requête...
📊 Comparaison avec 6 thèmes...

✅ Top 1 thèmes les plus pertinents :

1. [0.7924] Détection de fraude bancaire en temps réel
   Domaine : Machine Learning, Détection d'anomalies, Finance, Sécurité


🔍 Recherche pour : 'Projet sur la classification d'images médicales'
⏳ Génération de l'embedding de la requête...
📊 Comparaison avec 6 thèmes...

✅ Top 1 thèmes les plus pertinents :

1. [0.7032] Classification d'images médicales avec CNN
   Domaine : Deep Learning, Computer Vision, Santé, CNN


📊 ÉVALUATION DU SYSTÈME
Taux de réussite : 100.0%

1. ✅ Score: 0.6272
   Question : Je veux 

[{'question': 'Je veux faire un chatbot intelligent',
  'attendu': 'Chatbot RAG',
  'obtenu': 'Chatbot RAG pour support client automatisé',
  'score': 0.6272202709818272,
  'succes': True},
 {'question': 'Comment détecter les fraudes dans les banques?',
  'attendu': 'Détection de fraude',
  'obtenu': 'Détection de fraude bancaire en temps réel',
  'score': 0.7924200133654334,
  'succes': True},
 {'question': "Projet sur la classification d'images médicales",
  'attendu': "Classification d'images médicales",
  'obtenu': "Classification d'images médicales avec CNN",
  'score': 0.7031783881466176,
  'succes': True}]